# 介紹

本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。

In [68]:
import pyautogui as pyg
pyg.sleep(2)
basex=77
basey=290
gap=40

In [69]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)

0 0 77 290 (250, 27, 55)
0 1 117 290 (229, 229, 229)
0 2 157 290 (241, 15, 241)
0 3 197 290 (233, 103, 33)
0 4 237 290 (233, 103, 33)
0 5 277 290 (15, 164, 33)
0 6 317 290 (250, 27, 55)
0 7 357 290 (15, 164, 33)
1 0 77 330 (233, 103, 33)
1 1 117 330 (253, 245, 36)
1 2 157 330 (241, 15, 241)
1 3 197 330 (15, 164, 33)
1 4 237 330 (229, 229, 229)
1 5 277 330 (17, 140, 253)
1 6 317 330 (233, 103, 33)
1 7 357 330 (253, 200, 15)
2 0 77 370 (253, 245, 36)
2 1 117 370 (250, 27, 55)
2 2 157 370 (233, 103, 33)
2 3 197 370 (241, 15, 241)
2 4 237 370 (229, 229, 229)
2 5 277 370 (229, 229, 229)
2 6 317 370 (233, 103, 33)
2 7 357 370 (233, 103, 33)
3 0 77 410 (233, 103, 33)
3 1 117 410 (253, 245, 36)
3 2 157 410 (241, 15, 241)
3 3 197 410 (233, 103, 33)
3 4 237 410 (241, 15, 241)
3 5 277 410 (17, 140, 253)
3 6 317 410 (241, 15, 241)
3 7 357 410 (250, 27, 55)
4 0 77 450 (229, 229, 229)
4 1 117 450 (17, 140, 253)
4 2 157 450 (233, 103, 33)
4 3 197 450 (233, 103, 33)
4 4 237 450 (15, 164, 33)
4 5 277 4

In [70]:
def gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心點的螢幕座標 (x, y)。"""
    return basex + c * gap, basey + r * gap

In [71]:
print(gem_position(0,0))
print(gem_position(0,1))
print(gem_position(1,0))

(77, 290)
(117, 290)
(77, 330)


In [72]:
im = pyg.screenshot()
for i in range(8):
    x, y = gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, 1,im))

(229, 229, 229)
(229, 229, 229)
(229, 229, 229)
(229, 229, 229)
(229, 229, 229)
(229, 229, 229)
(229, 229, 229)
(229, 229, 229)


In [73]:
def get_gem_color(r, c, im):
    """回傳截圖中第 r 列、第 c 欄寶石中心點的顏色。"""
    x, y = gem_position(r, c)
    return im.getpixel((x, y))

In [74]:
def can_move_left(r, c, im):
    """判斷 (r, c) 的寶石左移後是否會形成三個以上相同顏色的連線。"""
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    current_color = get_gem_color(r, c, im)
    left_color = get_gem_color(r, c - 1, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return left_color
        if (row, col) == (r, c - 1):
            return current_color
        return get_gem_color(row, col, im)

    for center_r, center_c in ((r, c), (r, c - 1)):
        for dr, dc in ((0, 1), (1, 0)):
            for start in (-2, -1, 0):
                positions = [
                    (center_r + (start + i) * dr, center_c + (start + i) * dc)
                    for i in range(3)
                ]
                if all(0 <= row < 8 and 0 <= col < 8 for row, col in positions):
                    colors = [color_after_swap(row, col) for row, col in positions]
                    if colors[0] == colors[1] == colors[2]:
                        return True

    return False

In [75]:
im = pyg.screenshot()
print(can_move_left(2, 3,im))

True


In [ ]:
def _can_move_in_direction(r, c, im, dr, dc):
    """判斷交換指定方向相鄰寶石後，是否會形成三連線。"""
    next_r, next_c = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= next_r < 8 and 0 <= next_c < 8):
        return False

    current_color = get_gem_color(r, c, im)
    next_color = get_gem_color(next_r, next_c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return next_color
        if (row, col) == (next_r, next_c):
            return current_color
        return get_gem_color(row, col, im)

    for center_r, center_c in ((r, c), (next_r, next_c)):
        for match_dr, match_dc in ((0, 1), (1, 0)):
            for start in (-2, -1, 0):
                positions = [
                    (center_r + (start + i) * match_dr, center_c + (start + i) * match_dc)
                    for i in range(3)
                ]
                if all(0 <= row < 8 and 0 <= col < 8 for row, col in positions):
                    colors = [color_after_swap(row, col) for row, col in positions]
                    if colors[0] == colors[1] == colors[2]:
                        return True

    return False

def can_move_right(r, c, im):
    """判斷 (r, c) 的寶石右移後是否會形成三連線。"""
    return _can_move_in_direction(r, c, im, 0, 1)

def can_move_up(r, c, im):
    """判斷 (r, c) 的寶石上移後是否會形成三連線。"""
    return _can_move_in_direction(r, c, im, -1, 0)

def can_move_down(r, c, im):
    """判斷 (r, c) 的寶石下移後是否會形成三連線。"""
    return _can_move_in_direction(r, c, im, 1, 0)

In [ ]:
import time

start_time = time.monotonic()
im = pyg.screenshot()
directions = (
    (can_move_left, 0, -1),
    (can_move_right, 0, 1),
    (can_move_up, -1, 0),
    (can_move_down, 1, 0),
)

while True:
    if time.monotonic() - start_time >= 60:
        break

    moved = False
    for r in range(8):
        if time.monotonic() - start_time >= 60:
            break
        for c in range(8):
            if time.monotonic() - start_time >= 60:
                break
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = gem_position(r, c)
                    x2, y2 = gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.2, button="left")
                    pyg.sleep(0.2)
                    im = pyg.screenshot()
                    moved = True
                    break
            if moved:
                break
        if moved:
            break

    if not moved:
        pyg.sleep(0.1)
        im = pyg.screenshot()